# Black-Box Optimisation — per-function analysis

This notebook reproduces the approach described in the project's
[model card](../model-card.md): a Gaussian Process surrogate with per-dimension
length scales (ARD), the kernel chosen per function between RBF and Matern on
leave-one-out error, UCB/EI acquisition to propose the next query, and
leave-one-out diagnostics to judge each fit.

All eight functions are maximisation problems on the unit hypercube. Run the
cells top to bottom; the shared logic lives in `../src/bbo.py`.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
import bbo

DATA = "../data/history"
FUNCTIONS = range(1, 9)

## Fit each function and report diagnostics

For each function we load its query history, pick the kernel by leave-one-out
RMSE, fit the GP, and report the leave-one-out RMSE, the mean absolute z-score
(calibration; around 0.8 is well-calibrated), the best value seen so far, and
the round it was hit.

In [ ]:
rows = []
models = {}
for f in FUNCTIONS:
    X, y = bbo.load_function(f"{DATA}/function_{f}.csv")
    dims = X.shape[1]
    kernel, _ = bbo.choose_kernel(X, y)
    gp = bbo.make_gp(dims, kernel=kernel).fit(X, y)
    rmse, mz = bbo.loocv(X, y, kernel=kernel)
    best, best_round = float(np.max(y)), int(np.argmax(y)) + 1
    models[f] = (X, y, dims, kernel, gp, best)
    rows.append((f, dims, kernel, rmse, mz, best, best_round))

print(f"{'Fn':>3} {'Dim':>3} {'kernel':>7} {'LOO-RMSE':>10} {'mean|z|':>12} {'best':>12} {'round':>6}")
for f, dims, kernel, rmse, mz, best, br in rows:
    print(f"{f:>3} {dims:>3} {kernel:>7} {rmse:>10.4g} {mz:>12.3g} {best:>12.4g} {br:>6}")

## Best-so-far by round

The per-round outputs (faint) against the running best (solid). A few functions
step up sharply (F5, F7); others stay flat or noisy (F1, and F2 only late).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for f, ax in zip(FUNCTIONS, axes.ravel()):
    X, y, dims, kernel, gp, best = models[f]
    running = np.maximum.accumulate(y)
    ax.plot(range(1, len(y)+1), y, "o-", alpha=0.4, label="per round")
    ax.plot(range(1, len(y)+1), running, "s-", label="best so far")
    ax.set_title(f"Function {f} ({dims}D)")
    ax.set_xlabel("round"); ax.set_ylabel("output"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Propose the next query

UCB tracks a confirmed region; EI looks just beyond the current best. Both
candidates are printed per function, clipped to the cube and formatted to six
decimals (the portal's submission format). The final choice between them is made
per function, as described in the model card.

In [ ]:
for f in FUNCTIONS:
    X, y, dims, kernel, gp, best = models[f]
    q_ucb = bbo.propose_next(gp, dims, acq="ucb", beta=1.0)
    q_ei  = bbo.propose_next(gp, dims, acq="ei", y_best=best, xi=0.01)
    print(f"F{f} ({dims}D)")
    print(f"   UCB: {bbo.format_query(q_ucb)}")
    print(f"   EI : {bbo.format_query(q_ei)}")

## Notes

- F1's outputs sit near zero throughout, so its GP is effectively flat and its
  calibration score is meaningless; this matches the model card's note that its
  peak is too sharp for the current sampling to resolve.
- The higher-dimensional functions (F6-F8) have only a handful of points
  relative to their volume, so their fits are indicative rather than reliable.
- Seeds are fixed in `bbo.py`, so re-running gives the same numbers.